In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [90]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import skew
from sklearn.decomposition import PCA, KernelPCA
from xgboost import XGBRegressor

In [91]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.linear_model import ElasticNet, SGDRegressor, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBRegressor

In [92]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [93]:
train=pd.read_csv('./X_train.csv')
test=pd.read_csv('./X_test.csv')
price_res=pd.read_csv('./y_train.csv')
 

# data preprocessing

In [ ]:
# train .columns
# for col in train.columns:
#     print(col,train[col].dtype)

In [ ]:
# full = train

### deal with years

In [ ]:
# # Convert '建築完成年月' to datetime format
# full['建築完成年月'] = pd.to_datetime(full['建築完成年月'])

# # Get the current date
# today = datetime.today()

# df = pd.DataFrame()


# df['date_str'] = full['交易年'].astype(str) + '-' + full['交易月'].astype(str).str.zfill(2) + '-' + full['交易日'].astype(str).str.zfill(2)
# df['交易日期'] = pd.to_datetime(df['date_str'])
# #ˋ計算交易時屋齡
# full['建築年紀']=df['交易日期'].dt.year- full['建築完成年月'].dt.year


# print(full['建築年紀'])
# #
# full['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)

# print(full['交易距今時間'])


#### drop original trade and build date

In [ ]:
# full = full.drop(['交易年', '交易日', '交易月','建築完成年月'], axis=1)

### encode

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# labelencoder = LabelEncoder()

# for col in full.columns:
#     if full[col].dtype=='object':
#         full[col] = labelencoder.fit_transform(train[col])


In [ ]:
# import pandas as pd
# from datetime import datetime
# from sklearn.pipeline import Pipeline, FeatureUnion
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import FunctionTransformer, LabelEncoder
# from sklearn.base import BaseEstimator, TransformerMixin

# class DateTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         pass

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         df = X.copy()
#         df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
#         today = datetime.today()
#         df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
#         df['交易日期'] = pd.to_datetime(df['date_str'])
#         df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
#         df['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)
#         df = df.drop(['交易年', '交易月', '交易日', '建築完成年月', 'date_str', '交易日期'], axis=1)
#         return df

# class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         self.label_encoders = {}

#     def fit(self, X, y=None):
#         for col in X.select_dtypes(include=['object']).columns:
#             le = LabelEncoder()
#             le.fit(X[col])
#             self.label_encoders[col] = le
#         return self

#     def transform(self, X):
#         df = X.copy()
#         for col, le in self.label_encoders.items():
#             df[col] = le.transform(df[col])
#         return df

# # Create the pipeline
# pipeline = Pipeline([
#     ('date_transformer', DateTransformer()),
#     ('label_encoder', LabelEncoderTransformer())
# ])

# # Apply the pipeline to your data
# full = train.copy()
# full_transformed = pipeline.fit_transform(full)

# print(full_transformed)


# pipeline

In [94]:
features_pre_drop =  [  '托兒所', '國中', '高中職', '大學',  '大賣場', '超市', '百貨公司']
 

In [177]:
import pandas as pd
from datetime import datetime
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, LabelEncoder,OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

class DateTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
        today = datetime.today()
        df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
        df['交易日期'] = pd.to_datetime(df['date_str'])
        df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
        df = df.drop(['交易年', '交易月', '交易日', '建築完成年月', 'date_str','交易日期'], axis=1)
        #df = df.drop(['托兒所', '國中', '高中職', '大學',  '大賣場', '超市', '百貨公司'], axis=1)
        return df

class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.label_encoders[col] = le
        return self

    def transform(self, X):
        df = X.copy()
        for col, le in self.label_encoders.items():
            df[col] = le.transform(df[col])
        return df
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, handle_unknown='ignore'):
        """
        Initializes the OneHotEncoderTransformer.

        Args:
            sparse (bool, default=False): Whether to return a sparse matrix
                or a dense array. Sparse matrices can be more memory-efficient
                for large datasets, but may be slower to process.
            handle_unknown (str, default='ignore'): Specifies how to handle
                unknown categories during transformation. Possible values are:
                - 'error': Raise an error for unknown categories.
                - 'ignore': Ignore unknown categories and treat them as
                  previously unseen categories.
        """
        self.encoders = {}
        self.handle_unknown = handle_unknown

    def fit(self, X, y=price_res['單價元平方公尺']):
        """
        Fits the transformer to the data X.

        Args:
            X (pd.DataFrame): The data to fit the transformer on.
            y (None, optional): Not used in this context.

        Returns:
            OneHotEncoderTransformer: The fitted transformer.
        """
        for col in X.select_dtypes(include=['object']).columns:
            encoder = OneHotEncoder( handle_unknown=self.handle_unknown)
            encoder.fit(X[[col]])  # Reshape to 2D for OneHotEncoder
            self.encoders[col] = encoder
        return self

    def transform(self, X):
        """
        Transforms the data X using one-hot encoding.

        Args:
            X (pd.DataFrame): The data to transform.

        Returns:
            pd.DataFrame: The transformed data with one-hot encoded columns.
        """
        encoded_df = pd.DataFrame()
        for col, encoder in self.encoders.items():
            encoded_features = encoder.transform(X[[col]]).toarray()
            encoded_df = pd.concat([encoded_df, pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out([col]))], axis=1)
        return encoded_df

class GaussianBasisFunctionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, ngrid=3):
        self.ngrid = ngrid

    def fit(self, X, y=None):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        xmin, xmax = min(coord_x), max(coord_x) + 0.1
        ymin, ymax = min(coord_y), max(coord_y) + 0.1

        self.xgrids = np.linspace(xmin, xmax, self.ngrid + 1)
        self.ygrids = np.linspace(ymin, ymax, self.ngrid + 1)

        self.mu_x_all, self.mu_y_all, self.std_x_all, self.std_y_all, self.count_all = [], [], [], [], []

        for i in range(self.ngrid):
            for j in range(self.ngrid):
                x1, x2 = self.xgrids[i], self.xgrids[i + 1]
                y1, y2 = self.ygrids[j], self.ygrids[j + 1]
                tmpindx = (x1 <= coord_x) * (coord_x < x2)
                tmpindy = (y1 <= coord_y) * (coord_y < y2)
                tmpind = tmpindx * tmpindy
                npoints = np.sum(tmpind)

                if npoints >= 20:
                    mu_x = np.mean(coord_x[tmpind])
                    mu_y = np.mean(coord_y[tmpind])
                    std_x = np.std(coord_x[tmpind])
                    std_y = np.std(coord_y[tmpind])

                    self.mu_x_all.append(mu_x)
                    self.mu_y_all.append(mu_y)
                    self.std_x_all.append(std_x)
                    self.std_y_all.append(std_y)
                    self.count_all.append(npoints)

        return self

    def transform(self, X):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        ngf = len(self.mu_x_all)
        gf_all = np.zeros((coord_x.shape[0], ngf))

        for ii in range(ngf):
            mu_x = self.mu_x_all[ii]
            mu_y = self.mu_y_all[ii]
            std_x = self.std_x_all[ii]
            std_y = self.std_y_all[ii]

            tmpgf = np.exp(-(coord_x - mu_x) ** 2 / (2 * std_x ** 2) - (coord_y - mu_y) ** 2 / (2 * std_y ** 2))
            gf_all[:, ii] = tmpgf

        gf_df = pd.DataFrame(gf_all, columns=[f"gf_{i}" for i in range(gf_all.shape[1])])
        return pd.concat([X, gf_df], axis=1)

# Create the pipeline
pipeline = Pipeline([
    ('drop_columns', DropColumns(columns=[ 'Id'])),
    ('date_transformer', DateTransformer()),
    ('label_encoder', LabelEncoderTransformer()),
   #('onehot encoder',OneHotEncoderTransformer())
   ('gaussian_basis', GaussianBasisFunctionTransformer(ngrid=3))


])




In [178]:
# Apply the pipeline to your data
full = train.copy()
full_transformed = pipeline.fit_transform(full)
 


In [179]:
full_transformed.head(5)

,鄉鎮市區,交易標的,路名,土地移轉總面積平方公尺,都市土地使用分區,土地數,建物數,車位數,移轉層次,移轉層次項目,總樓層數,建物型態,主要用途,主要建材,建物移轉總面積平方公尺,建物現況格局-房,建物現況格局-廳,建物現況格局-衛,建物現況格局-隔間,有無管理組織,地鐵站,超商,公園,托兒所,國小,國中,高中職,大學,金融機構,醫院,大賣場,超市,百貨公司,警察局,消防局,縱坐標,橫坐標,建築年紀,gf_0,gf_1,gf_2,gf_3
0,9,1,472,27.75,0,1.0,1.0,1.0,1,6,7.0,5,15,6,133.43,3,2,2,0,0,1.0,7.0,2.0,20.0,20.0,19.0,12.0,17.0,15.0,20.0,7.0,20.0,13.0,20.0,16.0,24.957269,121.588026,0,0.006253,9.372051e-09,0.284171,5.820304e-19
1,1,0,546,9.57,19,1.0,1.0,0.0,5,6,6.0,5,4,6,40.34,1,1,1,0,0,1.0,12.0,8.0,20.0,20.0,20.0,13.0,20.0,20.0,20.0,13.0,20.0,20.0,20.0,20.0,24.997141,121.558262,24,0.998904,1.203964e-03,0.035053,4.981303e-08
2,9,0,426,9.51,0,1.0,1.0,0.0,1,6,7.0,3,4,6,70.61,1,1,1,0,0,0.0,6.0,7.0,20.0,20.0,18.0,11.0,19.0,15.0,18.0,6.0,20.0,11.0,20.0,13.0,24.953906,121.601050,11,0.000272,3.422241e-10,0.359073,5.674338e-20
3,3,1,181,23.67,19,1.0,1.0,1.0,10,6,15.0,0,15,6,143.83,3,2,2,0,0,1.0,12.0,14.0,20.0,20.0,14.0,5.0,4.0,19.0,20.0,12.0,20.0,13.0,20.0,15.0,25.008046,121.557424,10,0.819283,7.238877e-03,0.026498,5.412662e-06
4,4,0,77,22.50,19,1.0,1.0,0.0,2,6,5.0,1,4,6,83.73,2,1,1,0,1,2.0,4.0,13.0,20.0,20.0,15.0,5.0,15.0,19.0,20.0,10.0,20.0,16.0,18.0,13.0,24.986825,121.557424,48,0.831903,1.822802e-04,0.030968,1.251014e-10


In [180]:
full_transformed.columns,len(full_transformed.columns)

(Index(['鄉鎮市區', '交易標的', '路名', '土地移轉總面積平方公尺', '都市土地使用分區', '土地數', '建物數', '車位數',
        '移轉層次', '移轉層次項目', '總樓層數', '建物型態', '主要用途', '主要建材', '建物移轉總面積平方公尺',
        '建物現況格局-房', '建物現況格局-廳', '建物現況格局-衛', '建物現況格局-隔間', '有無管理組織', '地鐵站', '超商',
        '公園', '托兒所', '國小', '國中', '高中職', '大學', '金融機構', '醫院', '大賣場', '超市', '百貨公司',
        '警察局', '消防局', '縱坐標', '橫坐標', '建築年紀', 'gf_0', 'gf_1', 'gf_2', 'gf_3'],
       dtype='object'),
 42)

# gbf

In [ ]:
import numpy as np
import pandas as pd

def create_gaussian_basis_features(df, ngrid=3):
    """
    Creates Gaussian basis function features for a given DataFrame.

    Args:
        df: The DataFrame containing the coordinate data.
        ngrid: The number of grids to divide the space into.

    Returns:
        A DataFrame containing the original features and the new Gaussian basis function features.
    """

    coord_x = df['橫坐標'].values
    coord_y = df['縱坐標'].values

    xmin, xmax = min(coord_x), max(coord_x) + 0.1
    ymin, ymax = min(coord_y), max(coord_y) + 0.1

    xgrids = np.linspace(xmin, xmax, ngrid + 1)
    ygrids = np.linspace(ymin, ymax, ngrid + 1)

    mu_x_all, mu_y_all, std_x_all, std_y_all, count_all = [], [], [], [], []

    for i in range(ngrid):
        for j in range(ngrid):
            x1, x2 = xgrids[i], xgrids[i + 1]
            y1, y2 = ygrids[j], ygrids[j + 1]
            tmpindx = (x1 <= coord_x) * (coord_x < x2)
            tmpindy = (y1 <= coord_y) * (coord_y < y2)
            tmpind = tmpindx * tmpindy
            npoints = np.sum(tmpind)

            if npoints >= 20:
                mu_x = np.mean(coord_x[tmpind])
                mu_y = np.mean(coord_y[tmpind])
                std_x = np.std(coord_x[tmpind])
                std_y = np.std(coord_y[tmpind])

                mu_x_all.append(mu_x)
                mu_y_all.append(mu_y)
                std_x_all.append(std_x)
                std_y_all.append(std_y)
                count_all.append(npoints)

    def gaussian_basis(coord_x, coord_y, mu_x_all, mu_y_all, std_x_all, std_y_all):
        ngf = len(mu_x_all)
        gf_all = np.zeros((coord_x.shape[0], ngf))

        for ii in range(ngf):
            mu_x = mu_x_all[ii]
            mu_y = mu_y_all[ii]
            std_x = std_x_all[ii]
            std_y = std_y_all[ii]

            tmpgf = np.exp(-(coord_x - mu_x) ** 2 / (2 * std_x ** 2) - (coord_y - mu_y) ** 2 / (2 * std_y ** 2))
            gf_all[:, ii] = tmpgf

        return gf_all

    gf_all_train = gaussian_basis(coord_x, coord_y, mu_x_all, mu_y_all, std_x_all, std_y_all)
    gf_df = pd.DataFrame(gf_all_train, columns=[f"gf_{i}" for i in range(gf_all_train.shape[1])])

    return pd.concat([df, gf_df], axis=1)

# Example usage:
# full_transformed = create_gaussian_basis_features(full_transformed)
# print(full_transformed.columns)

Index(['鄉鎮市區', '交易標的', '路名', '土地移轉總面積平方公尺', '都市土地使用分區', '土地數', '建物數', '車位數',
       '移轉層次', '移轉層次項目', '總樓層數', '建物型態', '主要用途', '主要建材', '建物移轉總面積平方公尺',
       '建物現況格局-房', '建物現況格局-廳', '建物現況格局-衛', '建物現況格局-隔間', '有無管理組織', '地鐵站', '超商',
       '公園', '托兒所', '國小', '國中', '高中職', '大學', '金融機構', '醫院', '大賣場', '超市', '百貨公司',
       '警察局', '消防局', '縱坐標', '橫坐標', '建築年紀', 'gf_0', 'gf_1', 'gf_2', 'gf_3'],
      dtype='object')


# feature select

## corr 

In [205]:
import pandas as pd

def select_features(df1, df2, threshold=0.1):
    """
    Selects features from df2 based on correlation with a target column in df1.

    Args:
        df1: DataFrame containing the target column.
        df2: DataFrame containing potential features.
        threshold: Correlation threshold for feature selection.

    Returns:
        DataFrame: Filtered DataFrame with selected features.
    """

    correlation_series = df2.corrwith(df1['單價元平方公尺'])
    correlation_series = correlation_series.sort_values(ascending=False)

    features_to_drop = correlation_series[correlation_series.abs() < threshold].index.tolist()
    selected_df = df2.drop(features_to_drop, axis=1)

    return selected_df

In [206]:
full_selected=select_features(price_res,full_transformed)

In [ ]:
full_selected

In [207]:
len(full_selected.columns),len(full_transformed.columns)

(29, 42)

# model & Evaluate

In [105]:
from sklearn .model_selection import KFold

In [ ]:
# define cross validation strategy
def rmse_cv(model,X,y):
    
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kfold))
    return rmse


In [39]:
models = [RandomForestRegressor(n_jobs=-1),
          ExtraTreesRegressor(n_jobs=-1),XGBRegressor()]

In [ ]:
names = [ "RF","Extra","XGB"]
for name, model in zip(names, models):
    score = rmse_cv(model,full_transformed, price_res['單價元平方公尺'])
    print("{}: {:.6f}, {:.4f}".format(name,score.mean(),score.std()))

TypeError: got an unexpected keyword argument 'random_state'

In [129]:
names = [ "RF","Extra","XGB"]
for name, model in zip(names, models):
    score = rmse_cv(model,full_transformed, price_res['單價元平方公尺'])
    print("{}: {:.6f}, {:.4f}".format(name,score.mean(),score.std()))

RF: 33315.846234, 637.8822
Extra: 33253.168321, 956.2470
XGB: 33442.244887, 824.3270


In [ ]:
class grid():
    def __init__(self,model):
        self.model = model
    
    def grid_get(self,X,y,param_grid):
        grid_search = GridSearchCV(self.model,param_grid,cv=5, scoring="neg_mean_squared_error")
        grid_search.fit(X,y)
        print(grid_search.best_params_, np.sqrt(-grid_search.best_score_))
        grid_search.cv_results_['mean_test_score'] = np.sqrt(-grid_search.cv_results_['mean_test_score'])
        print(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score']])

### parameter
- xgb
  - 'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 1.0
- rf
  - 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20
- extra
  - 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300

# ensemble

In [41]:
class AverageWeight(BaseEstimator, RegressorMixin):
    def __init__(self,mod,weight):
        self.mod = mod
        self.weight = weight
        
    def fit(self,X,y):
        self.models_ = [clone(x) for x in self.mod]
        for model in self.models_:
            model.fit(X,y)
        return self
    
    def predict(self,X):
        w = list()
        pred = np.array([model.predict(X) for model in self.models_])
        # for every data point, single model prediction times weight, then add them together
        for data in range(pred.shape[1]):
            single = [pred[model,data]*weight for model,weight in zip(range(pred.shape[0]),self.weight)]
            w.append(np.sum(single))
        return w

In [194]:
xgb=XGBRegressor(colsample_bytree= 0.6, learning_rate= 0.1, max_depth= 8, min_child_weight= 1, subsample= 1.0)
rf=RandomForestRegressor(max_depth= None, max_features= 'sqrt', min_samples_leaf= 1, min_samples_split= 2, n_estimators= 20,n_jobs=-1)
extra=ExtraTreesRegressor(max_depth= None, min_samples_leaf= 2, min_samples_split= 2, n_estimators= 300,n_jobs=-1)

In [161]:
# assign weights based on their gridsearch score
w1 = 0.3

w2 = 0.35
w3 = 0.35



In [162]:
weight_avg = AverageWeight(mod = [rf,xgb,extra],weight=[w1,w2,w3])

## score 

In [163]:
score =rmse_cv(weight_avg,full_selected,price_res['單價元平方公尺']),  rmse_cv(weight_avg,full_selected,price_res['單價元平方公尺']).mean()
score

(array([33080.18604398, 31866.67384215, 32578.36391714, 32198.17534256,
        33602.67304278]),
 np.float64(32717.548009212594))

## fit

In [ ]:
weight_avg.fit(full_selected,price_res['單價元平方公尺'])


In [ ]:

# Apply the pipeline to your data
test = test.copy()
test_transformed = pipeline.fit_transform(test)
test_selected = test_transformed.drop(features_to_drop, axis=1)
#print(test_transformed)

In [ ]:
res=weight_avg.predict(test_selected)

In [ ]:

 
# 建立 DataFrame
df = pd.DataFrame({'單價元平方公尺': res})

# 新增 Id 欄位
# Create a DataFrame, prioritizing 'Id'
df = pd.DataFrame({'Id': df.index, '單價元平方公尺': res})

# Specify the output directory
output_dir = './output'  # Replace with your desired path
output_file = output_dir + '/output.csv'



# # 將 DataFrame 輸出為 CSV 檔案
df.to_csv(output_file, index=False)

# stacking


In [143]:
from sklearn.ensemble import StackingRegressor 

In [164]:
stack=StackingRegressor([rf,extra,xgb],[rf])

In [ ]:
rg = Ridge()
stack = StackingRegressor(
    estimators=[('rf', rf), ('extra', extra), ('xgb', xgb)],
    final_estimator=rg  # Use the separate XGBRegressor instance
)


In [196]:
score =rmse_cv(stack,full_selected,price_res['單價元平方公尺']),  rmse_cv(stack,full_selected,price_res['單價元平方公尺']).mean()
score

(array([32921.46048963, 31795.77603888, 32813.47953205, 32337.29044773,
        33459.68256855]),
 np.float64(32651.359879356634))

In [197]:
stack.fit(full_selected,price_res['單價元平方公尺'])


StackingRegressor(estimators=[('rf',
                               RandomForestRegressor(max_features='sqrt',
                                                     n_estimators=20,
                                                     n_jobs=-1)),
                              ('extra',
                               ExtraTreesRegressor(min_samples_leaf=2,
                                                   n_estimators=300,
                                                   n_jobs=-1)),
                              ('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.6, device=None,
                                            early_stopping_rounds=Non...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.1, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=8,
                                            max_leaves=None, min_child_weight=1,
                                            missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...))],
                  final_estimator=Ridge())

In [198]:

# Apply the pipeline to your data
test = test.copy()
test_transformed = pipeline.fit_transform(test)
test_selected = test_transformed.drop(features_to_drop, axis=1)
#print(test_transformed)

In [199]:
test_selected.shape

(2366, 29)

In [200]:
res=stack.predict(test_selected)


In [201]:

 
# 建立 DataFrame
df = pd.DataFrame({'單價元平方公尺': res})

# 新增 Id 欄位
# Create a DataFrame, prioritizing 'Id'
df = pd.DataFrame({'Id': df.index, '單價元平方公尺': res})

# Specify the output directory
output_dir = './output'  # Replace with your desired path
output_file = output_dir + '/output.csv'



# # 將 DataFrame 輸出為 CSV 檔案
df.to_csv(output_file, index=False)